# 04 — Deep Learning (Comparison Only)

A simple neural network for crowd bucket classification using scikit-learn's MLPClassifier.

> **Note**: This is a comparison model only. The production model uses RandomForestClassifier.
> TensorFlow/Keras is not required — we use scikit-learn's MLP to demonstrate the neural-network approach.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

from apps.predict.ml.features import engineer_features, REVERSE_BUCKET_MAP
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/ahmedabad_metro_bookings.csv')
if 'bucket' not in df.columns:
    def bucket_crowd(v):
        if v <= 50: return 'Low'
        elif v <= 150: return 'Medium'
        return 'High'
    df['bucket'] = df['actual_crowd'].apply(bucket_crowd)

X = engineer_features(df)
y = df['bucket'].map(REVERSE_BUCKET_MAP)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Features: {X.shape[1]}, Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# MLP Neural Network — 2 hidden layers
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    verbose=True,
)

mlp.fit(X_train_s, y_train)
print(f'\nTest accuracy: {mlp.score(X_test_s, y_test):.4f}')
print(f'Best validation score: {mlp.best_validation_score_:.4f}')

In [ ]:
# Training loss curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mlp.loss_curve_, color='#6366f1', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('MLP Training Loss Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
y_pred = mlp.predict(X_test_s)
print('Classification Report (MLP):')
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High'], zero_division=0))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Low', 'Medium', 'High'], ax=ax, cmap='Purples')
ax.set_title('Confusion Matrix — MLPClassifier')
plt.tight_layout()
plt.show()

In [ ]:
# Compare with production model
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_s, y_train)

print(f'RandomForest (production) accuracy: {rf.score(X_test_s, y_test):.4f}')
print(f'MLPClassifier (comparison) accuracy: {mlp.score(X_test_s, y_test):.4f}')
print(f'\nConclusion: The RandomForest model is used in production for its interpretability and feature importance analysis.')